# 04. Field-of-View (FOV) Optimization Demonstration

This notebook demonstrates the principles behind optimizing telescope pointing and position angle (PA) for improved differential photometry. We'll simulate star fields and an instrument's field of view to illustrate how the system identifies the best pointing to maximize comparison star coverage.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

# --- Configuration ---
TARGET_RA = 15.0 # degrees
TARGET_DEC = 30.0 # degrees
FOV_RADIUS_ARCSEC = 300 # Arcseconds (e.g., for a 10 arcmin field of view)
NUM_STARS_IN_FIELD = 50
OPTIMAL_STAR_MAG_RANGE = (12, 16) # Magnitude range for good comparison stars
# Simple representation of instrument footprint (e.g., square)
INSTRUMENT_WIDTH_ARCSEC = 600 # 10 arcmin
INSTRUMENT_HEIGHT_ARCSEC = 600 # 10 arcmin

print("Starting FOV Optimization Demonstration...")


## 1. Simulate Star Field and Instrument Footprint

We'll generate a mock star field around a target and define a simplified instrument footprint. In a real scenario, star data would come from Gaia DR3 and the footprint from instrument XML definitions.

In [ ]:
# Generate random stars around the target
np.random.seed(42) # for reproducibility

star_ras = TARGET_RA + np.random.uniform(-FOV_RADIUS_ARCSEC/3600, FOV_RADIUS_ARCSEC/3600, NUM_STARS_IN_FIELD)
star_decs = TARGET_DEC + np.random.uniform(-FOV_RADIUS_ARCSEC/3600, FOV_RADIUS_ARCSEC/3600, NUM_STARS_IN_FIELD)
star_mags = np.random.uniform(10, 20, NUM_STARS_IN_FIELD) # Random magnitudes

# Identify 'good' comparison stars within a magnitude range
good_comparison_stars_mask = (star_mags >= OPTIMAL_STAR_MAG_RANGE[0]) & (star_mags <= OPTIMAL_STAR_MAG_RANGE[1])

def plot_fov(ax, ra_center, dec_center, pa_deg, stars_ra, stars_dec, stars_mags, 
             instrument_width, instrument_height, title=""): 
    # Convert instrument dimensions to degrees
    width_deg = instrument_width / 3600.0
    height_deg = instrument_height / 3600.0

    # Define corners of the rectangular FOV centered at (ra_center, dec_center)
    # Points are relative to center, then rotated
    half_width = width_deg / 2
    half_height = height_deg / 2
    
    corners_unrotated = np.array([
        [-half_width, -half_height],
        [ half_width, -half_height],
        [ half_width,  half_height],
        [-half_width,  half_height]
    ])

    # Rotate corners by PA
    # Note: RA rotation needs to account for cos(dec)
    pa_rad = np.deg2rad(pa_deg)
    cos_pa, sin_pa = np.cos(pa_rad), np.sin(pa_rad)
    
    rotated_corners = np.zeros_like(corners_unrotated)
    rotated_corners[:, 0] = corners_unrotated[:, 0] * cos_pa - corners_unrotated[:, 1] * sin_pa
    rotated_corners[:, 1] = corners_unrotated[:, 0] * sin_pa + corners_unrotated[:, 1] * cos_pa

    # Shift to target center
    fov_ra_corners = ra_center + rotated_corners[:, 0] / np.cos(np.deg2rad(dec_center))
    fov_dec_corners = dec_center + rotated_corners[:, 1]

    # Plotting
    ax.scatter(stars_ra, stars_decs, c=stars_mags, cmap='viridis_r', s=50, zorder=2, label='Field Stars')
    ax.scatter(stars_ra[good_comparison_stars_mask], stars_decs[good_comparison_stars_mask], 
               marker='o', facecolors='none', edgecolors='green', s=100, linewidth=1.5, zorder=3, label='Good Comparison Stars')
    ax.plot(TARGET_RA, TARGET_DEC, 'X', color='red', markersize=10, label='Science Target', zorder=4)

    fov_polygon = Polygon(np.column_stack([fov_ra_corners, fov_dec_corners]), 
                          closed=True, edgecolor='red', facecolor='red', alpha=0.1, lw=2, zorder=1)
    ax.add_patch(fov_polygon)
    ax.plot(ra_center, dec_center, '+', color='purple', markersize=10, label='FOV Center', zorder=4)

    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_aspect('equal', adjustable='box')

plt.figure(figsize=(10, 10))
ax = plt.gca()
plot_fov(ax, TARGET_RA, TARGET_DEC, 0, star_ras, star_decs, star_mags,
         INSTRUMENT_WIDTH_ARCSEC, INSTRUMENT_HEIGHT_ARCSEC, 
         title="Initial FOV (PA=0, Centered on Target)")
plt.show()


## 2. Define Optimization Metric

The goal of FOV optimization is to maximize the number of 'good' comparison stars within the instrument's footprint, while ensuring the science target remains observable. A simple metric could be the count of good comparison stars in the FOV.

In [ ]:
def count_stars_in_fov(ra_center, dec_center, pa_deg, stars_ra, stars_dec, instrument_width, instrument_height, target_ra, target_dec, target_margin_arcsec=10):
    # Convert instrument dimensions and target margin to degrees
    width_deg = instrument_width / 3600.0
    height_deg = instrument_height / 3600.0
    target_margin_deg = target_margin_arcsec / 3600.0

    # Define corners of the rectangular FOV relative to its center, then rotate
    half_width = width_deg / 2
    half_height = height_deg / 2

    corners_unrotated = np.array([
        [-half_width, -half_height],
        [ half_width, -half_height],
        [ half_width,  half_height],
        [-half_width,  half_height]
    ])
    
    pa_rad = np.deg2rad(pa_deg)
    cos_pa, sin_pa = np.cos(pa_rad), np.sin(pa_rad)
    
    rotated_corners_x = corners_unrotated[:, 0] * cos_pa - corners_unrotated[:, 1] * sin_pa
    rotated_corners_y = corners_unrotated[:, 0] * sin_pa + corners_unrotated[:, 1] * cos_pa

    # Star coordinates relative to FOV center
    delta_ra = (stars_ra - ra_center) * np.cos(np.deg2rad(dec_center))
    delta_dec = (stars_dec - dec_center)

    # Inverse rotate stars to align with unrotated FOV box for easier checking
    inv_pa_rad = -pa_rad
    cos_inv_pa, sin_inv_pa = np.cos(inv_pa_rad), np.sin(inv_pa_rad)

    stars_in_fov_x = delta_ra * cos_inv_pa - delta_dec * sin_inv_pa
    stars_in_fov_y = delta_ra * sin_inv_pa + delta_dec * cos_inv_pa

    # Check if stars are within the unrotated rectangular bounds
    in_fov_mask = ((stars_in_fov_x >= -half_width) & (stars_in_fov_x <= half_width) &
                   (stars_in_fov_y >= -half_height) & (stars_in_fov_y <= half_height))

    # Ensure target is also within a central margin (e.g., to avoid vignetting)
    target_delta_ra = (target_ra - ra_center) * np.cos(np.deg2rad(dec_center))
    target_delta_dec = (target_dec - dec_center)

    target_rot_x = target_delta_ra * cos_inv_pa - target_delta_dec * sin_inv_pa
    target_rot_y = target_delta_ra * sin_inv_pa + target_delta_dec * cos_inv_pa

    target_in_margin = ((target_rot_x >= -half_width + target_margin_deg) &
                       (target_rot_x <= half_width - target_margin_deg) &
                       (target_rot_y >= -half_height + target_margin_deg) &
                       (target_rot_y <= half_height - target_margin_deg))

    if not target_in_margin:
        return -1 # Penalize heavily if target is not in safe margin

    return np.sum(in_fov_mask & good_comparison_stars_mask)

# Test the metric
initial_score = count_stars_in_fov(TARGET_RA, TARGET_DEC, 0, star_ras, star_decs, 
                                   INSTRUMENT_WIDTH_ARCSEC, INSTRUMENT_HEIGHT_ARCSEC, TARGET_RA, TARGET_DEC)
print(f"Initial score (centered, PA=0): {initial_score} good comparison stars.")


## 3. Simple Grid Search Optimization

We'll perform a basic grid search over a range of pointing offsets (dRA, dDec) and Position Angles (PA) to find the combination that maximizes our score.

In [ ]:
# Define search space
d_ra_offsets_arcsec = np.linspace(-100, 100, 10) # from -100 to +100 arcsec
d_dec_offsets_arcsec = np.linspace(-100, 100, 10)
pa_angles_deg = np.linspace(0, 359, 20) # 0 to 359 degrees

best_score = -1
best_offset_ra = 0
best_offset_dec = 0
best_pa = 0

for dra_arcsec in d_ra_offsets_arcsec:
    for ddec_arcsec in d_dec_offsets_arcsec:
        for pa in pa_angles_deg:
            current_ra_center = TARGET_RA + dra_arcsec / 3600.0 / np.cos(np.deg2rad(TARGET_DEC))
            current_dec_center = TARGET_DEC + ddec_arcsec / 3600.0
            
            score = count_stars_in_fov(current_ra_center, current_dec_center, pa,
                                       star_ras, star_decs, INSTRUMENT_WIDTH_ARCSEC, INSTRUMENT_HEIGHT_ARCSEC,
                                       TARGET_RA, TARGET_DEC)
            
            if score > best_score:
                best_score = score
                best_offset_ra = dra_arcsec
                best_offset_dec = ddec_arcsec
                best_pa = pa

print("\nOptimization Results:")
print(f"  Best Score: {best_score} good comparison stars")
print(f"  Optimal RA Offset: {best_offset_ra:.2f} arcsec")
print(f"  Optimal Dec Offset: {best_offset_dec:.2f} arcsec")
print(f"  Optimal Position Angle: {best_pa:.2f} degrees")


## 4. Visualize Optimal FOV

Plot the star field with the instrument's footprint placed at the optimal pointing and position angle.

In [ ]:
optimal_ra_center = TARGET_RA + best_offset_ra / 3600.0 / np.cos(np.deg2rad(TARGET_DEC))
optimal_dec_center = TARGET_DEC + best_offset_dec / 3600.0

plt.figure(figsize=(10, 10))
ax = plt.gca()
plot_fov(ax, optimal_ra_center, optimal_dec_center, best_pa, star_ras, star_decs, star_mags,
         INSTRUMENT_WIDTH_ARCSEC, INSTRUMENT_HEIGHT_ARCSEC,
         title=f"Optimal FOV (RA Offset: {best_offset_ra:.1f}\', Dec Offset: {best_offset_dec:.1f}\', PA: {best_pa:.1f}deg)")
plt.show()


## 5. Summary

This notebook provided a conceptual demonstration of FOV optimization, simulating a star field and finding an optimal telescope pointing and position angle to maximize the number of good comparison stars. In the `muscat-db` system, this process is automated and integrated with real Gaia DR3 data and instrument specifications to aid in observation planning.